In [ ]:
!pip install pythainlp scikit-learn pandas numpy

In [ ]:
import pandas as pd
import joblib
import time
from sklearn.metrics import accuracy_score, classification_report

# --- 1. ฟังก์ชันตัวช่วย (ห้ามลบ) ---
def find_column(df, target):
    """ค้นหาชื่อคอลัมน์ที่ใกล้เคียงที่สุดในไฟล์"""
    for col in df.columns:
        if target.lower() in col.lower().strip():
            return col
    return None

def map_rating_to_label(rating):
    """แปลงคะแนน 1-5 เป็นกลุ่ม Sentiment (เฉลย)"""
    try:
        r = float(rating)
        if r >= 4: return 'Positive'
        if r == 3: return 'Neutral'
        return 'Negative'
    except:
        return 'Neutral'

def run_accuracy_report(file_name, tfidf_model, sentiment_model, label_name):
    print(f"\n{'='*50}")
    print(f"📊 เริ่มต้นประมวลผลข้อมูล: {label_name}")
    print(f"{'='*50}")

    start_time = time.time()

    # อ่านไฟล์ข้อมูล (โหลดทั้งหมด 500,000 แถว)
    print(f"⏳ กำลังโหลดไฟล์ {file_name}...")
    df = pd.read_csv(base_path + file_name)

    # ค้นหาคอลัมน์ q9 และข้อความ
    q_col = find_column(df, 'q9')
    t_col = find_column(df, 'feedback_text')

    if not q_col or not t_col:
        print(f"❌ Error: หาคอลัมน์ q9 หรือข้อความใน {file_name} ไม่เจอ")
        return

    print(f"✅ พบข้อมูล: ใช้ '{q_col}' เป็นเฉลย และ '{t_col}' เป็นข้อความ")

    # 1. สร้างเฉลย (Ground Truth)
    y_true = df[q_col].apply(map_rating_to_label)

    # 2. ให้ AI ทำนายผล (Prediction)
    print(f"🔮 AI กำลังวิเคราะห์ข้อมูล {len(df):,} รายการ... (ขั้นตอนนี้ใช้เวลานาน)")
    X_tfidf = tfidf_model.transform(df[t_col].fillna(''))
    y_pred = sentiment_model.predict(X_tfidf)

    # 3. สรุปผล
    acc = accuracy_score(y_true, y_pred)
    elapsed = (time.time() - start_time) / 60

    print(f"\n✨ ผลลัพธ์สำหรับ {label_name}:")
    print(f"🎯 ความแม่นยำ (Accuracy): {acc * 100:.2f}%")
    print(f"⏱️ เวลาที่ใช้: {elapsed:.2f} นาที")
    print("\n📝 รายละเอียดความแม่นยำรายกลุ่ม:")
    print(classification_report(y_true, y_pred))

# ==========================================
# 🚀 เริ่มการทำงานรวม 1,000,000 Records
# ==========================================

# โหลดสมอง AI (ใช้ตัวเดียวกับที่ปันเทรนไว้)
print("📥 กำลังโหลดสมอง AI เข้าสู่ระบบ...")
tfidf = joblib.load(base_path + 'my_tfidf_opd.pkl')
model = joblib.load(base_path + 'my_sentiment_model_opd.pkl')

# รันส่วน OPD (500,000 แถว)
run_accuracy_report('suth_hospital_opd_500k.csv', tfidf, model, "OPD Analytics")

# รันส่วน IPD (500,000 แถว)
run_accuracy_report('suth_hospital_ipd_500k.csv', tfidf, model, "IPD Analytics")

print("\n🏆 ประมวลผลข้อมูลครบ 1,000,000 รายการเรียบร้อยแล้ว!")

📥 กำลังโหลดสมอง AI เข้าสู่ระบบ...

📊 เริ่มต้นประมวลผลข้อมูล: OPD Analytics
⏳ กำลังโหลดไฟล์ suth_hospital_opd_500k.csv...
✅ พบข้อมูล: ใช้ 'q9_overall_rating' เป็นเฉลย และ 'feedback_text' เป็นข้อความ
🔮 AI กำลังวิเคราะห์ข้อมูล 500,000 รายการ... (ขั้นตอนนี้ใช้เวลานาน)


KeyboardInterrupt: 

📥 กำลังโหลดไฟล์ข้อมูลใหม่...


FileNotFoundError: [Errno 2] No such file or directory: 'test_data_100k.csv'

In [ ]:
import pandas as pd
import time
import joblib
from google.colab import drive
from pythainlp.tokenize import word_tokenize


print("📂 กำลังเชื่อมต่อกับ Google Drive...")
drive.mount('/content/drive')
base_path = '/content/drive/MyDrive/SUTH_BigData/'


def thai_tokenizer(text):
    return word_tokenize(text, engine='newmm')


print("\n กำลังปลุกสมอง AI จาก Google Drive...")
try:
    tfidf_opd = joblib.load(base_path + 'my_tfidf_opd.pkl')
    ensemble_model_opd = joblib.load(base_path + 'my_sentiment_model_opd.pkl')
    tfidf_ipd = joblib.load(base_path + 'my_tfidf_ipd.pkl')
    ensemble_model_ipd = joblib.load(base_path + 'my_sentiment_model_ipd.pkl')
    print(" ปลุกสมอง AI สำเร็จ! พร้อมทำงานครับ")
except Exception as e:
    print(f"โหลดสมอง AI ไม่สำเร็จ (ตรวจเช็คไฟล์ใน Drive): {e}")


force_negative_keywords = [
    "รอนาน", "นานมาก", "ช้า", "เสียเวลา", "รอคิวนาน", "ล่าช้า", "รอจนรากงอก",
    "แพง", "ราคาแพง", "ขูดรีด", "น้อยมาก", "ไม่พอ",
    "แย่", "หยาบ", "พูดจาไม่ดี", "พูดจาแย่", "ไล่คนไข้", "ไม่ใส่ใจ", "ไม่มีมารยาท",
    "ส้นตีน", "ชิบหาย", "ห่วยแตก", "แย่มาก", "แย่สุด", "หมา", "กาก", "กวนตีน", "สถุล", "วีน", "เหวี่ยง", "ตะคอก", "ชักสีหน้า",
    "สกปรก", "ไม่สะอาด", "เหม็น", "แคบ", "ร้อน", "ไม่มีที่จอด",
    "จ่ายยาผิด", "จัดยาผิด", "วินิจฉัยผิด", "ให้ยาผิด", "ผิดพลาด", "อันตราย", "กินยาผิด",
    "ปรับปรุง", "ไม่ประทับใจ", "ผิดหวัง", "ร้องเรียน", "ห่วย", "ปรับปรุงด่วน"
]

def get_hybrid_sentiment(text, ml_result):
    if pd.isna(text): return ml_result
    for word in force_negative_keywords:
        if word in str(text):
            return "Negative"
    return ml_result


category_keywords = {
    " แพทย์และการรักษา (Doctor)": ["หมอ", "แพทย์", "นพ", "พญ", "ตรวจ", "รักษา", "วินิจฉัย", "มือเบา", "อาจารย์หมอ", "เลี้ยงไข้", "หมอเวร", "ผ่าตัด"],
    "พยาบาลและผู้ช่วย (Nurse)": ["พยาบาล", "ผู้ช่วย", "ซักประวัติ", "หน้าห้องตรวจ", "เจาะเลือด", "วัดความดัน", "พี่พยาบาล", "ชุดขาว", "จุดคัดกรอง"],
    "เภสัชกรและห้องยา (Pharmacy)": ["ยา", "เภสัช", "จ่ายยา", "ห้องยา", "อธิบายยา", "รับยา", "ช่องรับยา"],
    " การเงินและเวชระเบียน (Cashier)": ["เงิน", "จ่ายเงิน", "ราคา", "แพง", "บิล", "สิทธิ", "ประกัน", "บัตรทอง", "ห้องบัตร", "เวชระเบียน", "ทะเบียน", "แคชเชียร์", "ขูดรีด", "ใบเสร็จ", "ทำประวัติ"],
    " ระบบคิวและเวลา (Queue)": ["คิว", "นาน", "ช้า", "เร็ว", "เสียเวลา", "รอนาน", "รอจนรากงอก", "ล่าช้า", "แป๊บเดียว", "ข้ามคิว", "แซงคิว", "รอ", "ทั้งวัน"],
    " อาคารและที่จอดรถ (Parking)": ["ที่จอดรถ", "ลานจอด", "วนรถ", "ทางเดิน", "ลิฟต์", "ป้าย", "ทางเข้า", "ไม่มีที่จอด", "จอด", "อาคาร", "ตึก"],
    " สิ่งอำนวยความสะดวก (Comfort)": ["แอร์", "ร้อน", "หนาว", "เตียง", "หมอน", "ผ้าห่ม", "ห้องพัก", "เสียงดัง", "ที่นั่ง", "เก้าอี้", "ทีวี", "ตู้เย็น", "ระเบียง"],
    " ความสะอาดและสุขอนามัย (Cleanliness)": ["สกปรก", "เหม็น", "สะอาด", "ถังขยะ", "แม่บ้าน", "ถูพื้น", "ขี้นก", "กลิ่น", "ห้องน้ำ", "ฝุ่น", "เน่า"],
    " รปภ. (Security)": ["รปภ", "ยาม", "คนโบกรถ", "โบกรถ", "รักษาความปลอดภัย", "จราจร"],
    " กริยามารยาทและการบริการ (Manners)": ["พูดจา", "บริการ", "ดูแล", "สุภาพ", "หยาบ", "หยาบคาย", "มารยาท", "ด่า", "ตะคอก", "ชักสีหน้า", "เหวี่ยง", "วีน", "หน้าหงิก", "สถุล", "ส้นตีน", "ชิบหาย", "ห่วยแตก", "กาก", "แย่มาก", "ไล่คนไข้", "ไม่ใส่ใจ", "หมา", "กวนตีน", "กระแทกเสียง"]
}

def extract_categories(text):
    found_categories = []
    for category, keywords in category_keywords.items():
        if any(keyword in str(text) for keyword in keywords):
            found_categories.append(category)
    if not found_categories:
        return [" ทั่วไป (General Feedback)"]
    return found_categories


def predict_smart_feedback(text, mode):
    print(f"\n🔍 กำลังวิเคราะห์ข้อความ: '{text}' (แผนก: {mode})...")
    categories = extract_categories(text)
    tokens = word_tokenize(text, engine='newmm')
    processed_text = " ".join(tokens)

    if mode == 'OPD':
        text_vectorized = tfidf_opd.transform([processed_text])
        prediction = ensemble_model_opd.predict(text_vectorized)[0]
    else:
        text_vectorized = tfidf_ipd.transform([processed_text])
        prediction = ensemble_model_ipd.predict(text_vectorized)[0]

    prediction = get_hybrid_sentiment(text, prediction)

    if prediction == 'Positive': sentiment_result = "🟢 POSITIVE (เชิงบวก / คำชม)"
    elif prediction == 'Negative': sentiment_result = "🔴 NEGATIVE (เชิงลบ / คำติ)"
    else: sentiment_result = "⚪ NEUTRAL (เป็นกลาง / ทั่วไป)"

    print("-" * 60)
    print(f"📌 หมวดหมู่ที่เกี่ยวข้อง :")
    for cat in categories:
        print(f"   - {cat}")
    print(f"อารมณ์ของข้อความ   : {sentiment_result}")
    print("-" * 60)


def process_500k_data():
    print("\n=== 🚀 เริ่มกระบวนการวิเคราะห์ข้อมูล (โหมด Demo สำหรับพรีเซนต์ 5,000 แถว) ===")
    batch_start_time = time.time()

    def get_aspects_string(text):
        if pd.isna(text) or str(text).strip() == "": return "ทั่วไป (General Feedback)"
        return ", ".join(extract_categories(str(text)))

    try:
        print(f"\n[1/2] ⏳ กำลังประมวลผลไฟล์ OPD... (ดึงจาก Drive)")
        start_opd = time.time()
        df_opd_500k = pd.read_csv(base_path + 'suth_hospital_opd_500k.csv').head(5000)
        df_opd_500k['AI_Category'] = df_opd_500k['feedback_text'].apply(get_aspects_string)
        X_opd_500k = tfidf_opd.transform(df_opd_500k['feedback_text'].fillna(''))


        ml_sentiments_opd = ensemble_model_opd.predict(X_opd_500k)
        df_opd_500k['AI_Sentiment'] = [get_hybrid_sentiment(text, sent) for text, sent in zip(df_opd_500k['feedback_text'], ml_sentiments_opd)]

        df_opd_500k.to_csv(base_path + 'suth_hospital_opd_500k_ReadyForDashboard.csv', index=False, encoding='utf-8-sig')
        print(f"ประมวลผล OPD เสร็จสิ้น! ใช้เวลา {time.time() - start_opd:.2f} วินาที")

        print(f"\n[2/2] ⏳ กำลังประมวลผลไฟล์ IPD...")
        start_ipd = time.time()
        df_ipd_500k = pd.read_csv(base_path + 'suth_hospital_ipd_500k.csv').head(5000)
        df_ipd_500k['AI_Category'] = df_ipd_500k['feedback_text'].apply(get_aspects_string)
        X_ipd_500k = tfidf_ipd.transform(df_ipd_500k['feedback_text'].fillna(''))

        ml_sentiments_ipd = ensemble_model_ipd.predict(X_ipd_500k)
        df_ipd_500k['AI_Sentiment'] = [get_hybrid_sentiment(text, sent) for text, sent in zip(df_ipd_500k['feedback_text'], ml_sentiments_ipd)]

        df_ipd_500k.to_csv(base_path + 'suth_hospital_ipd_500k_ReadyForDashboard.csv', index=False, encoding='utf-8-sig')
        print(f"ประมวลผล IPD เสร็จสิ้น! ใช้เวลา {time.time() - start_ipd:.2f} วินาที")

        print("-" * 60)
        print(f"🎉 ประมวลผลเสร็จสมบูรณ์! ไฟล์ _ReadyForDashboard ส่งเข้า Google Drive แล้วครับ")
    except FileNotFoundError as e:
        print(f"\nไม่พบไฟล์ใน Drive ({e})")

def export_excel_by_category():
    print("\n===  ระบบดึงไฟล์ Excel คัดแยกประเภท ===")
    print("[1] OPD (ผู้ป่วยนอก) | [2] IPD (ผู้ป่วยใน)")
    dept_choice = input("เลือกแผนกที่ต้องการดึงข้อมูล (1 หรือ 2): ")

    if dept_choice == '1':
        file_name = 'suth_hospital_opd_500k_ReadyForDashboard.csv'
        dept_name = 'OPD'
    elif dept_choice == '2':
        file_name = 'suth_hospital_ipd_500k_ReadyForDashboard.csv'
        dept_name = 'IPD'
    else:
        print("⚠️ เลือกไม่ถูกต้อง ยกเลิกครับ")
        return

    keyword = input(f"พิมพ์คำหรือหมวดหมู่ที่ต้องการแยกจากแผนก {dept_name} (เช่น หมอ, คิว, แอร์, ที่จอดรถ): ")
    print(f"\n⏳ กำลังค้นหาคำว่า '{keyword}'...")

    try:
        df = pd.read_csv(base_path + file_name)
        mask = df['AI_Category'].str.contains(keyword, na=False) | df['feedback_text'].str.contains(keyword, na=False)
        filtered_df = df[mask]

        if len(filtered_df) > 0:
            export_name = f'Export_{dept_name}_{keyword}.xlsx'
            filtered_df.to_excel(base_path + export_name, index=False)
            print(f"จอข้อมูลทั้งหมด {len(filtered_df):,} แถว!")
            print(f"สร้างไฟล์ Excel ชื่อ '{export_name}' ลงใน Google Drive เรียบร้อย")
        else:
            print(f"ไม่พบข้อความที่เกี่ยวกับ '{keyword}' ")
    except FileNotFoundError:
        print(" หาไฟล์หลักไม่เจอ ต้องกดรันเมนู [3] ให้เสร็จก่อน")




print("=" * 60)
print("🚀 พร้อมทำงานแล้ว")
print("=" * 60)

while True:
    print("\n[1] ทดสอบแชท OPD | [2] ทดสอบแชท IPD | [3] รันข้อมูล Demo (5,000 แถว) | [4] ดึงไฟล์ Excel คัดแยกประเภท | [0] ออกจากระบบ")
    choice = input("เลือกเมนู (1, 2, 3, 4 หรือ 0): ")

    if choice == '0':
        print("ปิดระบบศูนย์บัญชาการ 👋")
        break
    elif choice in ['1', '2']:
        mode_name = 'OPD' if choice == '1' else 'IPD'
        user_input = input(f"💬 พิมพ์ข้อความที่ต้องการทดสอบสำหรับ {mode_name}: ")
        predict_smart_feedback(user_input, mode_name)
    elif choice == '3':
        process_500k_data()
    elif choice == '4':
        export_excel_by_category()
    else:
        print("⚠️ กรุณาพิมพ์ 1, 2, 3, 4 หรือ 0 เท่านั้น")

📂 กำลังเชื่อมต่อกับ Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

 กำลังปลุกสมอง AI จาก Google Drive...
 ปลุกสมอง AI สำเร็จ! พร้อมทำงานครับ
🚀 พร้อมทำงานแล้ว

[1] ทดสอบแชท OPD | [2] ทดสอบแชท IPD | [3] รันข้อมูล Demo (5,000 แถว) | [4] ดึงไฟล์ Excel คัดแยกประเภท | [0] ออกจากระบบ
เลือกเมนู (1, 2, 3, 4 หรือ 0): 1
💬 พิมพ์ข้อความที่ต้องการทดสอบสำหรับ OPD: พยาบาลพูดจาสุภาพ

🔍 กำลังวิเคราะห์ข้อความ: 'พยาบาลพูดจาสุภาพ' (แผนก: OPD)...
------------------------------------------------------------
📌 หมวดหมู่ที่เกี่ยวข้อง :
   - พยาบาลและผู้ช่วย (Nurse)
   - เภสัชกรและห้องยา (Pharmacy)
   -  กริยามารยาทและการบริการ (Manners)
อารมณ์ของข้อความ   : 🟢 POSITIVE (เชิงบวก / คำชม)
------------------------------------------------------------

[1] ทดสอบแชท OPD | [2] ทดสอบแชท IPD | [3] รันข้อมูล Demo (5,000 แถว) | [4] ดึงไฟล์ Excel คัดแยกประเภท | [0] ออกจากระบบ
เลือกเมนู (1, 2, 3, 4 หรือ 0): 4

===  ระบบดึงไฟล์ E

In [ ]:
import pandas as pd
import joblib
from google.colab import drive
from pythainlp.tokenize import word_tokenize


#  ส่วนที่เพิ่มเพื่อแก้ Error: สร้างฟังก์ชันตัดคำให้ AI รู้จักก่อนโหลด

def thai_tokenizer(text):
    return word_tokenize(text, engine='newmm')

# 1. เชื่อมต่อ Drive และกำหนด Path
drive.mount('/content/drive')
base_path = '/content/drive/MyDrive/SUTH_BigData/'

# 2. โหลด
print(" กำลังโหลด ...")
tfidf = joblib.load(base_path + 'my_tfidf_opd.pkl')
model = joblib.load(base_path + 'my_sentiment_model_opd.pkl')

# 3. อ่านไฟล์ Excel
print(" กำลังอ่านไฟล์ข้อมูลใหม่...")
file_name = 'ให้200.xlsx'
df = pd.read_excel(base_path + file_name)


#  ส่วนเพิ่ม 1: ตรวจคำในแง่ลบ (Hybrid Sentiment)

force_negative_keywords = [
    "รอนาน", "นานมาก", "ช้า", "เสียเวลา", "รอคิวนาน", "ล่าช้า", "นาน",
    "แพง", "ราคาแพง", "ขูดรีด", "น้อยมาก", "ไม่พอ",
    "แย่", "หยาบ", "พูดจาไม่ดี", "พูดจาแย่", "ไล่คนไข้", "ไม่ใส่ใจ", "ไม่มีมารยาท", "พูดเร็วไป",
    "ส้นตีน", "ชิบหาย", "ห่วยแตก", "แย่มาก", "แย่สุด", "หมา", "กาก",
    "สกปรก", "ไม่สะอาด", "เหม็น", "แคบ", "ร้อน", "ไม่มีที่จอด",
    "จ่ายยาผิด", "จัดยาผิด", "วินิจฉัยผิด", "ให้ยาผิด", "ผิดพลาด", "อันตราย", "กินยาผิด",
    "ปรับปรุง", "ไม่ประทับใจ", "ผิดหวัง", "ร้องเรียน", "ห่วย", "ปรับปรุงด่วน"
]

def get_hybrid_sentiment(text, ml_result):
    if pd.isna(text):
        return ml_result
    for word in force_negative_keywords:
        if word in str(text):
            return "Negative"
    return ml_result


#  ส่วนเพิ่ม 2: ระบบแนะนำการแก้ปัญหาอัตโนมัติ

solution_mapping = {
    "ความปลอดภัย (ด่วน)": {
        "keywords": ["จ่ายยาผิด", "จัดยาผิด", "วินิจฉัยผิด", "ให้ยาผิด", "ผิดพลาด", "อันตราย", "กินยาผิด"],
        "action": " [เร่งด่วน] ระงับเหตุ ตรวจสอบข้อผิดพลาดทางคลินิก (Incident Report) ทันที"
    },
    "พฤติกรรมบริการ": {
        "keywords": ["แย่", "หยาบ", "พูดจาไม่ดี", "พูดจาแย่", "ไล่คนไข้", "ไม่มีมารยาท", "ส้นตีน", "ชิบหาย", "ห่วยแตก", "กาก"],
        "action": " เรียกพนักงานตักเตือน/สอบสวน และจัดอบรมมาตรฐานการบริการ (Service Mind)"
    },
    "ระบบคิวและเวลา": {
        "keywords": ["รอนาน", "นานมาก", "ช้า", "เสียเวลา", "รอคิวนาน", "ล่าช้า", "คิว"],
        "action": " ตรวจสอบจุดคอขวด (Bottleneck) และพิจารณาเพิ่มบุคลากรในช่วงเวลาเร่งด่วน"
    },
    "ความสะอาดและสถานที่": {
        "keywords": ["สกปรก", "ไม่สะอาด", "เหม็น", "แคบ", "ร้อน", "แอร์", "ห้องน้ำ"],
        "action": " ประสานงานแม่บ้าน/ช่างอาคาร เข้าทำความสะอาดและตรวจสอบอุปกรณ์ทันที"
    },
    "ที่จอดรถ": {
        "keywords": ["ไม่มีที่จอด", "ที่จอดรถ", "ลานจอด", "น้อยมาก", "ไม่พอ"],
        "action": " จัด รปภ. อำนวยความสะดวกจราจร หรือพิจารณาเปิดพื้นที่จอดรถสำรอง"
    },
    "ราคาและค่าใช้จ่าย": {
        "keywords": ["แพง", "ราคาแพง", "ขูดรีด"],
        "action": " ตรวจสอบความถูกต้องของบิล และกำชับให้แจกแจงค่าใช้จ่ายก่อนให้บริการเสมอ"
    }
}

def get_suggested_solution(text, sentiment):
    if sentiment == 'Positive':
        return " นำไปชื่นชมบุคลากร และรักษามาตรฐานการบริการต่อไป"

    if pd.isna(text):
        return "-"

    found_actions = []
    for issue, data in solution_mapping.items():
        if any(keyword in str(text) for keyword in data["keywords"]):
            found_actions.append(data["action"])

    if not found_actions:
        return " หัวหน้างานควรตรวจสอบรายละเอียดเพิ่มเติมจากข้อความ"

    return " | ".join(found_actions)


#  ส่วน Keyword: แยกตัวละคร

personnel_keywords = {
    "แพทย์/หมอ": ["หมอ", "แพทย์", "นพ", "พญ", "นพ.", "พญ.", "คุณหมอ", "อาจารย์หมอ"],
    "พยาบาล": ["พยาบาล", "พี่พยาบาล", "คุณพยาบาล"],
    " แม่บ้าน/พนักงานทำความสะอาด": ["แม่บ้าน", "ทำความสะอาด", "คนทำความสะอาด", "คนถูพื้น", "ภารโรง","สกปรก"],
    "เจ้าหน้าที่เวชระเบียน/การเงิน": ["การเงิน", "เวชระเบียน", "ห้องบัตร", "คนเก็บเงิน", "แคชเชียร์", "ประชาสัมพันธ์"],
    " เภสัชกร": ["เภสัช", "คนจ่ายยา", "ห้องยา"],
    "รปภ.": ["รปภ", "ยาม", "รักษาความปลอดภัย", "คนโบกรถ", "ลานจอด"],
    "เวลา":["รอ","นาน","เวลา","ช้า","เร็ว", "คิว"],
    "สถานที่":["จอด","ห้องน้ำ","ที่นั่ง","แอร์","ตึก","หน้าต่าง","ประตู"],
    "ร้านค้า":["แพง","ร้าน","ของ","ของกิน"]
}

def extract_personnel(text):
    if pd.isna(text):
        return " ไม่ระบุเจาะจง (Unspecified)"
    found_persons = []
    for person, keywords in personnel_keywords.items():
        if any(keyword in str(text) for keyword in keywords):
            found_persons.append(person)
    if not found_persons:
        return " ไม่ระบุเจาะจง (Unspecified)"
    return ", ".join(found_persons)



print(" AI กำลังวิเคราะห์ข้อมูลและแก้ไขจุดผิดพลาด...")

# 4.1 ให้ AI ทำนายอารมณ์เบื้องต้น
X_new = tfidf.transform(df['feedback_text'].fillna(''))
df['AI_Sentiment'] = model.predict(X_new)

# 4.2 เช็คคำต้องห้าม (กฎเหล็กดักความแม่นยำ)
df['Sentiment'] = df.apply(lambda row: get_hybrid_sentiment(row['feedback_text'], row['AI_Sentiment']), axis=1)

# 4.3 แยกหมวดหมู่ตัวละคร
df['Target_Person'] = df['feedback_text'].apply(extract_personnel)

#  4.4 สร้างคอลัมน์แนวทางแก้ไข (แก้ไข: ใช้ค่าจาก Sentiment ที่ถูกต้อง)
print(" กำลังสร้างข้อเสนอแนะเพื่อการบริหารจัดการ...")
df['Suggested_Action'] = df.apply(lambda row: get_suggested_solution(row['feedback_text'], row['Sentiment']), axis=1)

#  บังคับจัดเรียงคอลัมน์ให้คลีน 100%
df = df[['feedback_text', 'Sentiment', 'Target_Person', 'Suggested_Action']]

# 5. บันทึกผลลัพธ์
export_name = file_name.replace('.xlsx', '_Result.xlsx')

df.to_excel(base_path + export_name, index=False)

print(f" เสร็จเรียบร้อย")
print(f" ไฟล์ถูกบันทึกไปที่: {base_path + export_name}")

print(f" เสร็จเรียบร้อย")
print(f"ไฟล์ถูกบันทึกไปที่: {base_path + export_name}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 กำลังโหลด ...
 กำลังอ่านไฟล์ข้อมูลใหม่...
 AI กำลังวิเคราะห์ข้อมูลและแก้ไขจุดผิดพลาด...
 กำลังสร้างข้อเสนอแนะเพื่อการบริหารจัดการ...
 เสร็จเรียบร้อย
 ไฟล์ถูกบันทึกไปที่: /content/drive/MyDrive/SUTH_BigData/ให้200_Result.xlsx
 เสร็จเรียบร้อย
ไฟล์ถูกบันทึกไปที่: /content/drive/MyDrive/SUTH_BigData/ให้200_Result.xlsx
